In [1]:
import pandas as pd
import re
from code import *
import tensorflow as tf
from tensorflow.keras.models import Model
import numpy as np
import os

In [2]:
os.chdir('../../')  # change dir to the work root DEDL-Kcr

In [3]:
 # 1.load_data
test_filepath = r'dataset/test_dataset.csv'
test_seqs, y_test = load_dataset(test_filepath)

# 2.extrac features
protein_bert_test = extract_embedding_features(test_seqs)
tf.keras.backend.clear_session()
BLOSUM62_test = BLOSUM62(test_seqs)
one_hot_test = np.array(to_embedding_numeric(test_seqs)).astype(np.float32)

In [4]:
model_1 = CNN(protein_bert_test)
model_2 = BiGRU()
model_3 = CNN(BLOSUM62_test)
model_atnn = ensemble_model()

In [5]:
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')

In [6]:
ensemble_valid_input = np.concatenate((model_1.predict(protein_bert_test, verbose=0),
                                  model_2.predict(one_hot_test, verbose=0),
                                  model_3.predict(BLOSUM62_test, verbose=0)), axis=-1)

In [7]:
outputs = 'code/experiment/results/ind_test'
prediction_result_ind = []
p_1 = model_atnn.predict(ensemble_valid_input, verbose=0)
tmp_result = np.zeros((len(y_test), 3))
tmp_result[:, 0], tmp_result[:, 1:] = y_test, p_1
prediction_result_ind.append(tmp_result)
metrics = save_val_result(prediction_result_ind, outputs, "Independent_test")

# Papaya dataset

In [65]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [66]:
def parse_sequences(path):
    fr = open(path, 'r')
    data = fr.readlines()
    seqs = data[1::2]
    return seqs

def clean(x):
    return x.strip()

In [67]:
papaya_pos = parse_sequences("dataset/papaya_31_neg.txt")
papaya_neg = parse_sequences("dataset/papaya_31_pos.txt")

In [68]:
papaya_pos = list(map(clean, papaya_pos))
papaya_neg = list(map(clean, papaya_neg))

In [69]:
seqs = papaya_pos + papaya_neg
y = len(papaya_pos) * [0] + len(papaya_neg) * [1]

In [70]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [71]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])

In [72]:
wef = extract_word_embedding_features(seqs, embedding_layer)

In [73]:
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)

In [74]:
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [76]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.7629
recall: 0.9082
f1_score: 0.793
precision_score: 0.7037
auc: 0.8486


# zebrafish dataset

In [3]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [11]:
zebrafish = pd.read_csv(r"dataset/zebrafish.csv", sep=",")
seqs = zebrafish[' Sequence']
y = [1] * len(seqs)

In [12]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [13]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])

In [14]:
wef = extract_word_embedding_features(seqs, embedding_layer)

In [ ]:
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)

In [21]:
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [27]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.7684
recall: 0.7684
f1_score: 0.869
precision_score: 1.0
auc: nan


D:\anaconda\envs\isumo-rsfpn\lib\site-packages\sklearn\metrics\_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


# Tea tree

In [29]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [32]:
tea = pd.read_csv(r"dataset/tea.csv", sep=",")
seqs = tea['Sequence']
y = tea['Label']

In [33]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [34]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])
wef = extract_word_embedding_features(seqs, embedding_layer)
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [35]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.909
recall: 0.909
f1_score: 0.9523
precision_score: 1.0
auc: nan


D:\anaconda\envs\isumo-rsfpn\lib\site-packages\sklearn\metrics\_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


# tobacco

In [36]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [38]:
tobacco = pd.read_csv(r"dataset/tobacco.csv", sep=",")
seqs = tobacco['Sequence']
y = tobacco['Label']

In [39]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [40]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])
wef = extract_word_embedding_features(seqs, embedding_layer)
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [41]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.8578
recall: 0.8578
f1_score: 0.9234
precision_score: 1.0
auc: nan


D:\anaconda\envs\isumo-rsfpn\lib\site-packages\sklearn\metrics\_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


# Lung Cancer

In [43]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [48]:
lung_cancer = pd.read_csv(r"dataset/lung cancer.csv", sep=",")
seqs = lung_cancer['Sequence']
y = [1] * len(seqs)

In [49]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [50]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])
wef = extract_word_embedding_features(seqs, embedding_layer)
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [51]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.898
recall: 0.898
f1_score: 0.9463
precision_score: 1.0
auc: nan


D:\anaconda\envs\isumo-rsfpn\lib\site-packages\sklearn\metrics\_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


# rice

In [77]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [78]:
rice = pd.read_csv(r"dataset/rice.csv", sep=",")
seqs = rice['Sequence']
y = rice['Label']

In [79]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [80]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])
wef = extract_word_embedding_features(seqs, embedding_layer)
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [81]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.7787
recall: 0.893
f1_score: 0.8329
precision_score: 0.7804
auc: 0.8376


# Pancreatic cancer

In [86]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

In [87]:
Pancreatic = pd.read_csv(r"dataset/Pancreatic cancer.csv", sep=",")
seqs = Pancreatic['Sequence']
y = [1] * len(seqs)

In [88]:
pbf = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
blosum62_f = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

In [89]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])
wef = extract_word_embedding_features(seqs, embedding_layer)
labels = np.array(y)
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = tf.concat([p_1, p_2, p_3], axis=1)
final_p = model_atnn.predict(p, verbose=0)
y_pred = np.zeros_like(final_p)
y_pred[final_p >= 0.5] = 1

In [90]:
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score, auc, roc_curve
acc = accuracy_score(labels, y_pred.flatten())
recall = recall_score(labels, y_pred.flatten())
f1_score = f1_score(labels, y_pred.flatten())
precision_score = precision_score(labels, y_pred.flatten())
fpr, tpr, thresholds = roc_curve(labels, final_p, pos_label=1)
auc = auc(fpr, tpr)
print("acc: {:.4}".format(acc))
print("recall: {:.4}".format(recall))
print("f1_score: {:.4}".format(f1_score))
print("precision_score: {:.4}".format(precision_score))
print("auc: {:.4}".format(auc))

acc: 0.863
recall: 0.863
f1_score: 0.9265
precision_score: 1.0
auc: nan


D:\anaconda\envs\isumo-rsfpn\lib\site-packages\sklearn\metrics\_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
